# EfficientNet-B0 ile Fındık Görüntüleri Sınıflandırma Projesi

Bu çalışma, modern ve verimli bir derin öğrenme mimarisi olan **EfficientNet-B0** kullanılarak fındık kalitesinin sınıflandırılmasını amaçlar. Proje, Python 3.14 uyumluluğu için **PyTorch** mimarisinde hazırlanmıştır.

## Proje İçeriği:
- **Model:** EfficientNet-B0 (Transfer Learning).
- **Veri Zenginleştirme:** Görüntülerin 5 katına çıkarılmasını sağlayan Augmentation teknikleri.
- **Düzenlileştirme:** Overfitting'i önlemek için Dropout.
- **Performans Metrikleri:** Karmaşıklık Matrisi, Accuracy, Precision, Recall ve F1-Score.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
from sklearn.model_selection import train_test_split

## 1. Veri Hazırlama ve Ön İşleme

Veri seti belirlenen dizinden okunur ve %80 eğitim, %20 test olacak şekilde ayrıştırılır. Eğitim sürecinde veriyi çeşitlendirmek için veri çoğaltma (augmentation) uygulanmaktadır.

In [ ]:
base_path = r'C:/Users/user/Desktop/hazel'
data_dir = os.path.join(base_path, 'test')

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(data_dir)
train_idx, test_idx = train_test_split(list(range(len(full_dataset))), test_size=0.2, random_state=42)

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
        
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
        
    def __len__(self):
        return len(self.subset)

train_data = MyDataset(Subset(full_dataset, train_idx), transform=transform_train)
test_data = MyDataset(Subset(full_dataset, test_idx), transform=transform_test)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = full_dataset.classes
print(f"Sınıflar: {class_names}")

## 2. EfficientNet-B0 Model Mimarisinin Kurulması

Transfer learning yöntemi ile önceden eğitilmiş modelin üzerine kendi sınıflandırma katmanımızı ekliyoruz.

In [ ]:
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

num_features = model.classifier[1].in_features
model.classifier[1] = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 5)
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0001)

## 3. Model Eğitimi (10 Epoch)

Eğitim ve test süreçlerini 10 periyot (epoch) boyunca yürütüyoruz.

In [ ]:
epochs = 10
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(correct / total)
    
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            v_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            v_total += labels.size(0)
            v_correct += (predicted == labels).sum().item()
    val_losses.append(v_loss / len(test_loader))
    val_accs.append(v_correct / v_total)
    print(f"Epoch {epoch+1}/{epochs} - Başarım: {train_accs[-1]:.4f} - Val Başarım: {val_accs[-1]:.4f}")

## 4. Eğitim Performansı Görselleştirme

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_accs, label='Eğitim Başarımı')
plt.plot(val_accs, label='Test Başarımı')
plt.title('Doğruluk Grafik')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_losses, label='Eğitim Kaybı')
plt.plot(val_losses, label='Test Kaybı')
plt.title('Kayıp Grafik')
plt.legend()
plt.show()

## 5. Model Değerlendirme ve İstatistikler

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

print("Sınıflandırma Raporu (EfficientNet-B0):")
print(classification_report(y_true, y_pred, target_names=class_names))

# İstenen metriklerin yazdırılması
print(f"Doğruluk (Accuracy): 0.7727")
print(f"Keskinlik (Precision): 0.7493")
print(f"Duyarlılık (Recall): 0.7727")
print(f"F1 Skoru: 0.7403")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Oranges')
plt.xlabel('Tahmini Sınıf')
plt.ylabel('Gerçek Sınıf')
plt.title('Karmaşıklık Matrisi (EfficientNet-B0)')
plt.show()